In [2]:
import psycopg2
from psycopg2 import OperationalError, sql
import pandas as pd
import requests
import logging

In [3]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

In [4]:
class DatabaseManager:
    def __init__(self, host, database, user, password, port):
        """Initialize database connection."""
        self.host = host
        self.database = database
        self.user = user
        self.password = password
        self.port = port
        self.connection = None
        self.cursor = None
        self.connect()

    def connect(self):
        """Establish database connection."""
        try:
            self.connection = psycopg2.connect(
                host=self.host,
                database=self.database,
                user=self.user,
                password=self.password,
                port=self.port
            )
            self.cursor = self.connection.cursor()
            logging.info("Connected to the database successfully.")
        except OperationalError as e:
            logging.error(f"Database connection failed: {e}")

    def create_table(self):
        """Create table if it doesn't exist."""
        query = """
            CREATE TABLE IF NOT EXISTS ml.jagan_country_table (
                id SERIAL PRIMARY KEY,
                name VARCHAR(100) UNIQUE NOT NULL,
                code CHAR(3) UNIQUE NOT NULL,
                continent VARCHAR(50) NOT NULL,
                population BIGINT
            );
        """
        self.execute_query(query, "Table ensured in the database.")

    def insert_from_csv(self, file_path):
        """Insert data from CSV using Pandas for batch insertion."""
        try:
            df = pd.read_csv(file_path)
            with self.connection.cursor() as cursor:
                for _, row in df.iterrows():
                    cursor.execute(
                        """
                        INSERT INTO ml.jagan_country_table (name, code, continent, population)
                        VALUES (%s, %s, %s, %s) 
                        ON CONFLICT (code) DO NOTHING;
                        """,
                        (row['name'], row['code'], row['continent'], row['population'])
                    )
            self.connection.commit()
            logging.info("CSV data inserted successfully.")
        except Exception as e:
            logging.error(f"Failed to insert data from CSV: {e}")

    def insert_country_data(self, name, code, continent, population):
        """Insert a single country's data."""
        query = """
            INSERT INTO ml.jagan_country_table (name, code, continent, population)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (code) DO NOTHING;
        """
        self.execute_query(query, "Inserted country data from API.", (name, code, continent, population))

    def fetch_all(self):
        """Fetch all records from the table."""
        query = "SELECT * FROM ml.jagan_country_table;"
        try:
            with self.connection.cursor() as cursor:
                cursor.execute(query)
                rows = cursor.fetchall()
                for row in rows:
                    logging.info(row)
        except Exception as e:
            logging.error(f"Failed to fetch records: {e}")

    def execute_query(self, query, success_message=None, params=None):
        """Execute a single SQL query safely with error handling."""
        try:
            with self.connection.cursor() as cursor:
                cursor.execute(query, params if params else ())
            self.connection.commit()
            if success_message:
                logging.info(success_message)
        except Exception as e:
            logging.error(f"SQL Execution Error: {e}")

    def close_connection(self):
        """Close the database connection."""
        if self.cursor:
            self.cursor.close()
        if self.connection:
            self.connection.close()
            logging.info("Database connection closed.")

In [5]:
class APIManager:

    API_URL = "https://restcountries.com/v3.1/all"

    def fetch_countries(self, limit=4):
        """Fetch country data from the API."""
        try:
            response = requests.get(self.API_URL, timeout=10)
            response.raise_for_status()
            return response.json()[:limit]
        except requests.RequestException as e:
            logging.error(f"API request failed: {e}")
            return []

In [6]:
# Database Credentials
DB_CONFIG = {
    "host": "ep-noisy-lake-a8k78ama-pooler.eastus2.azure.neon.tech",
    "database": "playground",
    "user": "jagan",
    "password": "SIT-ML-Playground",
    "port": 5432
}
#testing

db_manager = DatabaseManager(**DB_CONFIG)

db_manager.create_table()

csv_file_path = r"C:\Users\m_jag\OneDrive\Desktop\Projects\AI\country.csv"
db_manager.insert_from_csv(csv_file_path)

api_manager = APIManager()
countries = api_manager.fetch_countries(limit=4)

for country in countries:
    name = country["name"]["common"]
    code = country["cca2"]
    continent = country.get("region", "Unknown")
    population = country.get("population", 0)
    db_manager.insert_country_data(name, code, continent, population)

db_manager.fetch_all()

db_manager.close_connection()

2025-02-12 08:25:13,811 - INFO - Connected to the database successfully.
2025-02-12 08:25:14,483 - INFO - Table ensured in the database.
2025-02-12 08:25:15,118 - INFO - CSV data inserted successfully.
2025-02-12 08:25:17,226 - INFO - Inserted country data from API.
2025-02-12 08:25:17,854 - INFO - Inserted country data from API.
2025-02-12 08:25:18,495 - INFO - Inserted country data from API.
2025-02-12 08:25:19,144 - INFO - Inserted country data from API.
2025-02-12 08:25:19,563 - INFO - (1, 'india', 'IN ', 'Asia', 1380004385)
2025-02-12 08:25:19,565 - INFO - (2, 'South Georgia', 'GS ', 'Antarctic', 30)
2025-02-12 08:25:19,566 - INFO - (3, 'Grenada', 'GD ', 'Americas', 112519)
2025-02-12 08:25:19,567 - INFO - (7, 'Switzerland', 'CH ', 'Europe', 8654622)
2025-02-12 08:25:19,568 - INFO - (8, 'Sierra Leone', 'SL ', 'Africa', 7976985)
2025-02-12 08:25:19,570 - INFO - Database connection closed.
